In [112]:
from pathlib import Path
import ast

import numpy as np
import pandas as pd


project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent


processed_dir = (
    project_root
    / "data"
    / "processed"
)

processed_dir.mkdir(
    parents=True,
    exist_ok=True
)

project_root

WindowsPath('c:/Users/Dell - i5 11th Gen/Desktop/clinvar-gnomad-vus-analyzer')

In [113]:
matched_path = (
    processed_dir
    / "atm_clinvar_gnomad_enriched.csv"
)

unmatched_path = (
    processed_dir
    / "atm_clinvar_gnomad_unmatched.csv"
)


for file_path in [matched_path, unmatched_path]:
    if not file_path.exists():
        raise FileNotFoundError(
            f"Required file not found: {file_path}"
        )


identifier_dtypes = {
    "Variation": "string",
    "Genes": "string",
    "Protein change": "string",
    "VCV accession": "string",
    "rs_ID": "string",
    "variant_id": "string"
}


matched_variants = pd.read_csv(
    matched_path,
    dtype=identifier_dtypes
)

unmatched_variants = pd.read_csv(
    unmatched_path,
    dtype=identifier_dtypes
)


print("Matched variants:", len(matched_variants))
print("Unmatched variants:", len(unmatched_variants))
print(
    "Total variants:",
    len(matched_variants) + len(unmatched_variants)
)

Matched variants: 1474
Unmatched variants: 3060
Total variants: 4534


In [114]:
matched_variants["gnomad_match_status"] = "Matched"

unmatched_variants["gnomad_match_status"] = (
    "Not found in gnomAD"
)


print(
    matched_variants["gnomad_match_status"]
    .value_counts()
)

print(
    unmatched_variants["gnomad_match_status"]
    .value_counts()
)

gnomad_match_status
Matched    1474
Name: count, dtype: int64
gnomad_match_status
Not found in gnomAD    3060
Name: count, dtype: int64


In [115]:
gnomad_columns = [
    "gnomad_ac",
    "gnomad_an",
    "gnomad_af",
    "gnomad_homozygote_count",
    "gnomad_filters"
]


for column in gnomad_columns:
    if column not in unmatched_variants.columns:
        unmatched_variants[column] = pd.NA


unmatched_variants[
    [
        "variant_id",
        "gnomad_match_status",
        "gnomad_ac",
        "gnomad_an",
        "gnomad_af"
    ]
].head()

,variant_id,gnomad_match_status,gnomad_ac,gnomad_an,gnomad_af
0,11-108227628-A-C,Not found in gnomAD,<NA>,<NA>,<NA>
1,11-108227629-G-C,Not found in gnomAD,<NA>,<NA>,<NA>
2,11-108227630-T-A,Not found in gnomAD,<NA>,<NA>,<NA>
3,11-108227632-T-C,Not found in gnomAD,<NA>,<NA>,<NA>
4,11-108227637-C-T,Not found in gnomAD,<NA>,<NA>,<NA>


In [116]:
required_status_columns = [
    "gnomad_match_status",
    "gnomad_ac",
    "gnomad_an",
    "gnomad_af",
    "gnomad_homozygote_count",
    "gnomad_filters"
]


for column in required_status_columns:
    assert column in matched_variants.columns
    assert column in unmatched_variants.columns


print("Matched and unmatched columns are aligned.")

Matched and unmatched columns are aligned.


In [117]:
all_variants = pd.concat(
    [
        matched_variants,
        unmatched_variants
    ],
    ignore_index=True,
    sort=False
)


print("Combined rows:", len(all_variants))


all_variants[
    [
        "variant_id",
        "Protein change",
        "gnomad_match_status",
        "gnomad_af"
    ]
].head()

Combined rows: 4534


C:\Users\Dell - i5 11th Gen\AppData\Local\Temp\ipykernel_36468\1887789814.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_variants = pd.concat(


,variant_id,Protein change,gnomad_match_status,gnomad_af
0,11-108227628-A-G,S2G,Matched,0.000001
1,11-108227630-T-G,S2R,Matched,0.000001
2,11-108227634-G-T,V4L,Matched,0.000001
3,11-108227661-C-T,R13C,Matched,0.000011
4,11-108227662-G-A,R13H,Matched,0.000011


In [118]:
all_variants[
    "gnomad_match_status"
].value_counts()

gnomad_match_status
Not found in gnomAD    3060
Matched                1474
Name: count, dtype: int64

In [119]:
numeric_columns = [
    "Variation ID",
    "gnomad_ac",
    "gnomad_an",
    "gnomad_af",
    "gnomad_homozygote_count"
]


for column in numeric_columns:
    all_variants[column] = pd.to_numeric(
        all_variants[column],
        errors="coerce"
    )


all_variants["Variation ID"] = (
    all_variants["Variation ID"]
    .astype("Int64")
)


all_variants[numeric_columns].dtypes

Variation ID                 Int64
gnomad_ac                  float64
gnomad_an                  float64
gnomad_af                  float64
gnomad_homozygote_count    float64
dtype: object

In [120]:
all_variants["Germline date last evaluated"] = (
    pd.to_datetime(
        all_variants[
            "Germline date last evaluated"
        ],
        errors="coerce"
    )
)


all_variants[
    "Germline date last evaluated"
].head()

0   2024-02-19
1   2019-12-09
2   2025-01-12
3   2025-12-20
4   2026-01-26
Name: Germline date last evaluated, dtype: datetime64[ns]

In [121]:
required_genomic_columns = [
    "variant_id",
    "chromosome",
    "position",
    "reference_allele",
    "alternate_allele"
]


missing_columns = [
    column
    for column in required_genomic_columns
    if column not in all_variants.columns
]

if missing_columns:
    raise KeyError(
        f"Missing genomic columns: {missing_columns}"
    )


all_variants["chromosome"] = (
    all_variants["chromosome"]
    .astype("string")
)

all_variants["position"] = (
    pd.to_numeric(
        all_variants["position"],
        errors="coerce"
    )
    .astype("Int64")
)

all_variants["reference_allele"] = (
    all_variants["reference_allele"]
    .astype("string")
)

all_variants["alternate_allele"] = (
    all_variants["alternate_allele"]
    .astype("string")
)


expected_variant_id = (
    all_variants["chromosome"]
    + "-"
    + all_variants["position"].astype("string")
    + "-"
    + all_variants["reference_allele"]
    + "-"
    + all_variants["alternate_allele"]
)


variant_id_mismatch = (
    all_variants["variant_id"]
    .astype("string")
    .ne(expected_variant_id)
    .fillna(True)
)

if variant_id_mismatch.any():
    raise ValueError(
        "Some genomic columns do not match their variant_id."
    )


all_variants[
    required_genomic_columns
].head()

,variant_id,chromosome,position,reference_allele,alternate_allele
0,11-108227628-A-G,11,108227628,A,G
1,11-108227630-T-G,11,108227630,T,G
2,11-108227634-G-T,11,108227634,G,T
3,11-108227661-C-T,11,108227661,C,T
4,11-108227662-G-A,11,108227662,G,A


In [122]:
all_variants["transcript"] = (
    all_variants["Variation"]
    .str.extract(
        r"^(NM_\d+\.\d+)",
        expand=False
    )
    .astype("string")
)


all_variants["hgvs_c"] = (
    all_variants["Variation"]
    .str.extract(
        r":(c\.[^\s(]+)",
        expand=False
    )
    .astype("string")
)


all_variants["hgvs_p"] = (
    all_variants["Variation"]
    .str.extract(
        r"\((p\.[^)]+)\)",
        expand=False
    )
    .astype("string")
)


all_variants[
    [
        "Variation",
        "transcript",
        "hgvs_c",
        "hgvs_p"
    ]
].head()

,Variation,transcript,hgvs_c,hgvs_p
0,NM_000051.4(ATM):c.4A>G (p.Ser2Gly),NM_000051.4,c.4A>G,p.Ser2Gly
1,NM_000051.4(ATM):c.6T>G (p.Ser2Arg),NM_000051.4,c.6T>G,p.Ser2Arg
2,NM_000051.4(ATM):c.10G>T (p.Val4Leu),NM_000051.4,c.10G>T,p.Val4Leu
3,NM_000051.4(ATM):c.37C>T (p.Arg13Cys),NM_000051.4,c.37C>T,p.Arg13Cys
4,NM_000051.4(ATM):c.38G>A (p.Arg13His),NM_000051.4,c.38G>A,p.Arg13His


In [123]:
protein_parts = (
    all_variants["Protein change"]
    .str.extract(
        r"^([A-Z*])(\d+)([A-Z*])$"
    )
)


all_variants["reference_amino_acid"] = (
    protein_parts[0]
    .astype("string")
)

all_variants["protein_position"] = (
    pd.to_numeric(
        protein_parts[1],
        errors="coerce"
    )
    .astype("Int64")
)

all_variants["alternate_amino_acid"] = (
    protein_parts[2]
    .astype("string")
)


all_variants[
    [
        "Protein change",
        "reference_amino_acid",
        "protein_position",
        "alternate_amino_acid"
    ]
].head()

,Protein change,reference_amino_acid,protein_position,alternate_amino_acid
0,S2G,S,2,G
1,S2R,S,2,R
2,V4L,V,4,L
3,R13C,R,13,C
4,R13H,R,13,H


In [124]:
def format_gnomad_filters(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).strip()

    if text in {"", "nan", "<NA>"}:
        return pd.NA

    try:
        parsed_value = ast.literal_eval(text)
    except (ValueError, SyntaxError):
        return text

    if isinstance(parsed_value, list):
        if not parsed_value:
            return "PASS"

        return ", ".join(
            str(item)
            for item in parsed_value
        )

    return text


all_variants["gnomad_filters"] = (
    all_variants["gnomad_filters"]
    .apply(format_gnomad_filters)
    .astype("string")
)


all_variants[
    [
        "gnomad_match_status",
        "gnomad_filters"
    ]
].head()

,gnomad_match_status,gnomad_filters
0,Matched,PASS
1,Matched,PASS
2,Matched,PASS
3,Matched,PASS
4,Matched,PASS


In [125]:
app_columns = [
    "Genes",
    "Variation ID",
    "VCV accession",
    "rs_ID",
    "Variation",
    "transcript",
    "hgvs_c",
    "hgvs_p",
    "Protein change",
    "protein_position",
    "reference_amino_acid",
    "alternate_amino_acid",
    "variant_id",
    "chromosome",
    "position",
    "reference_allele",
    "alternate_allele",
    "Condition",
    "Classification",
    "Review status",
    "Molecular consequence",
    "Germline date last evaluated",
    "gnomad_match_status",
    "gnomad_ac",
    "gnomad_an",
    "gnomad_af",
    "gnomad_homozygote_count",
    "gnomad_filters"
]


app_dataset = (
    all_variants[app_columns]
    .copy()
)


app_dataset.head()

,Genes,Variation ID,VCV accession,rs_ID,Variation,transcript,hgvs_c,hgvs_p,Protein change,protein_position,...,Classification,Review status,Molecular consequence,Germline date last evaluated,gnomad_match_status,gnomad_ac,gnomad_an,gnomad_af,gnomad_homozygote_count,gnomad_filters
0,ATM,640846,VCV000640846,rs639367,NM_000051.4(ATM):c.4A>G (p.Ser2Gly),NM_000051.4,c.4A>G,p.Ser2Gly,S2G,2,...,G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,2024-02-19,Matched,2.0,1613596.0,0.000001,0.0,PASS
1,ATM,1337452,VCV001337452,rs1328461,NM_000051.4(ATM):c.6T>G (p.Ser2Arg),NM_000051.4,c.6T>G,p.Ser2Arg,S2R,2,...,G: Uncertain significance,"criteria provided, single submitter",missense variant,2019-12-09,Matched,2.0,1613628.0,0.000001,0.0,PASS
2,ATM,481224,VCV000481224,rs475312,NM_000051.4(ATM):c.10G>T (p.Val4Leu),NM_000051.4,c.10G>T,p.Val4Leu,V4L,4,...,G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,2025-01-12,Matched,2.0,1613478.0,0.000001,0.0,PASS
3,ATM,185977,VCV000185977,rs183070,NM_000051.4(ATM):c.37C>T (p.Arg13Cys),NM_000051.4,c.37C>T,p.Arg13Cys,R13C,13,...,G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,2025-12-20,Matched,18.0,1613346.0,0.000011,0.0,PASS
4,ATM,188171,VCV000188171,rs186132,NM_000051.4(ATM):c.38G>A (p.Arg13His),NM_000051.4,c.38G>A,p.Arg13His,R13H,13,...,G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,2026-01-26,Matched,17.0,1613566.0,0.000011,0.0,PASS


In [126]:
app_dataset = app_dataset.rename(
    columns={
        "Genes": "gene",
        "Variation ID": "clinvar_variation_id",
        "VCV accession": "vcv_accession",
        "rs_ID": "rs_id",
        "Variation": "clinvar_variation",
        "Protein change": "protein_change",
        "Condition": "condition",
        "Classification": "clinvar_classification",
        "Review status": "review_status",
        "Molecular consequence": "molecular_consequence",
        "Germline date last evaluated": (
            "clinvar_last_evaluated"
        )
    }
)


app_dataset.head()

,gene,clinvar_variation_id,vcv_accession,rs_id,clinvar_variation,transcript,hgvs_c,hgvs_p,protein_change,protein_position,...,clinvar_classification,review_status,molecular_consequence,clinvar_last_evaluated,gnomad_match_status,gnomad_ac,gnomad_an,gnomad_af,gnomad_homozygote_count,gnomad_filters
0,ATM,640846,VCV000640846,rs639367,NM_000051.4(ATM):c.4A>G (p.Ser2Gly),NM_000051.4,c.4A>G,p.Ser2Gly,S2G,2,...,G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,2024-02-19,Matched,2.0,1613596.0,0.000001,0.0,PASS
1,ATM,1337452,VCV001337452,rs1328461,NM_000051.4(ATM):c.6T>G (p.Ser2Arg),NM_000051.4,c.6T>G,p.Ser2Arg,S2R,2,...,G: Uncertain significance,"criteria provided, single submitter",missense variant,2019-12-09,Matched,2.0,1613628.0,0.000001,0.0,PASS
2,ATM,481224,VCV000481224,rs475312,NM_000051.4(ATM):c.10G>T (p.Val4Leu),NM_000051.4,c.10G>T,p.Val4Leu,V4L,4,...,G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,2025-01-12,Matched,2.0,1613478.0,0.000001,0.0,PASS
3,ATM,185977,VCV000185977,rs183070,NM_000051.4(ATM):c.37C>T (p.Arg13Cys),NM_000051.4,c.37C>T,p.Arg13Cys,R13C,13,...,G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,2025-12-20,Matched,18.0,1613346.0,0.000011,0.0,PASS
4,ATM,188171,VCV000188171,rs186132,NM_000051.4(ATM):c.38G>A (p.Arg13His),NM_000051.4,c.38G>A,p.Arg13His,R13H,13,...,G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,2026-01-26,Matched,17.0,1613566.0,0.000011,0.0,PASS


In [127]:
app_dataset = (
    app_dataset
    .sort_values(
        by=[
            "protein_position",
            "position",
            "alternate_allele"
        ],
        ascending=True,
        na_position="last"
    )
    .reset_index(drop=True)
)


app_dataset[
    [
        "protein_change",
        "protein_position",
        "variant_id",
        "gnomad_match_status",
        "gnomad_af"
    ]
].head(10)

,protein_change,protein_position,variant_id,gnomad_match_status,gnomad_af
0,S2R,2,11-108227628-A-C,Not found in gnomAD,NaN
1,S2G,2,11-108227628-A-G,Matched,0.000001
2,S2T,2,11-108227629-G-C,Not found in gnomAD,NaN
3,S2R,2,11-108227630-T-A,Not found in gnomAD,NaN
4,S2R,2,11-108227630-T-G,Matched,0.000001
5,L3P,3,11-108227632-T-C,Not found in gnomAD,NaN
6,V4L,4,11-108227634-G-T,Matched,0.000001
7,L5V,5,11-108227637-C-G,Not found in gnomAD,NaN
8,L5F,5,11-108227637-C-T,Not found in gnomAD,NaN
9,L5R,5,11-108227638-T-G,Not found in gnomAD,NaN


In [128]:
matched_app_rows = app_dataset[
    app_dataset["gnomad_match_status"]
    == "Matched"
].copy()


unmatched_app_rows = app_dataset[
    app_dataset["gnomad_match_status"]
    == "Not found in gnomAD"
].copy()


print("Matched app rows:", len(matched_app_rows))
print("Unmatched app rows:", len(unmatched_app_rows))

Matched app rows: 1474
Unmatched app rows: 3060


In [129]:
expected_total = len(matched_variants) + len(unmatched_variants)

assert len(app_dataset) == expected_total
assert app_dataset["variant_id"].notna().all()
assert app_dataset["variant_id"].is_unique
assert app_dataset["clinvar_variation_id"].notna().all()

print("Final app dataset validation passed.")

Final app dataset validation passed.


In [130]:
gnomad_numeric_columns = [
    "gnomad_ac",
    "gnomad_an",
    "gnomad_af",
    "gnomad_homozygote_count"
]

assert (
    matched_app_rows[gnomad_numeric_columns]
    .notna()
    .all()
    .all()
)

assert (
    unmatched_app_rows[gnomad_numeric_columns]
    .isna()
    .all()
    .all()
)

print("gnomAD columns survived Phase 4 correctly.")

gnomAD columns survived Phase 4 correctly.


In [131]:
protein_parse_failures = app_dataset[
    app_dataset["protein_position"].isna()
]


print(
    "Protein changes needing review:",
    len(protein_parse_failures)
)


protein_parse_failures[
    [
        "protein_change",
        "hgvs_p",
        "variant_id"
    ]
].head(20)

Protein changes needing review: 8


,protein_change,hgvs_p,variant_id
4526,<NA>,p.Leu146_Lys147delinsHisAsn,11-108235775-TCAAA-ACAAT
4527,<NA>,p.Phe627_Gln628delinsLeuLys,11-108252895-CC-AA
4528,<NA>,p.Met660_Asp661delinsLysAsn,11-108253894-TGG-AGA
4529,<NA>,p.Leu961_Pro962delinsPheAla,11-108271108-GC-TG
4530,<NA>,p.Met1210_Ala1211delinsIleSer,11-108282763-GG-TT
4531,<NA>,p.Met1321_Leu1322delinsIleIle,11-108284443-GC-AA
4532,<NA>,p.Leu1419_Leu1420delinsProPhe,11-108289621-TTC-CTT
4533,<NA>,p.Asn1739_Ile1740delinsThrPhe,11-108301686-ACA-CCT


In [132]:
app_dataset["protein_parse_status"] = np.where(
    app_dataset["protein_position"].notna(),
    "Parsed",
    "Needs review"
)

In [133]:
app_output_path = (
    processed_dir
    / "atm_vus_app_dataset.csv"
)


app_dataset.to_csv(
    app_output_path,
    index=False,
    date_format="%Y-%m-%d"
)


print(
    "App dataset saved to:",
    app_output_path
)

App dataset saved to: c:\Users\Dell - i5 11th Gen\Desktop\clinvar-gnomad-vus-analyzer\data\processed\atm_vus_app_dataset.csv


In [134]:
parsed_protein_count = (
    app_dataset["protein_parse_status"]
    .eq("Parsed")
    .sum()
)

needs_review_count = (
    app_dataset["protein_parse_status"]
    .eq("Needs review")
    .sum()
)


print("\nPhase 4 summary")
print("----------------------")
print("Total app variants:", len(app_dataset))
print("Matched in gnomAD:", len(matched_app_rows))
print(
    "Not found in gnomAD:",
    len(unmatched_app_rows)
)
print("App columns:", len(app_dataset.columns))
print(
    "Protein changes parsed:",
    parsed_protein_count
)
print(
    "Protein changes needing review:",
    needs_review_count
)
print()
print("App dataset:", app_output_path)


Phase 4 summary
----------------------
Total app variants: 4534
Matched in gnomAD: 1474
Not found in gnomAD: 3060
App columns: 29
Protein changes parsed: 4526
Protein changes needing review: 8

App dataset: c:\Users\Dell - i5 11th Gen\Desktop\clinvar-gnomad-vus-analyzer\data\processed\atm_vus_app_dataset.csv
